In [50]:
from meshing import Mesh2D
import matplotlib.pyplot as plt
from problems import Problem, continuous_coefficient_2d, constant_coefficient_2d
from discretization import discretize
import numpy as np
import scipy.sparse as sps
import pandas as pd
import tqdm

In [51]:
class CG:
    name = "CG"

    def __init__(
        self,
        preconditioner=None,
        rtol: float = 1e-7,
        atol: float = 0,
        maxiter: int | None = 1000,
        estimate_cond: bool = False,
    ):
        self.preconditioner = preconditioner
        self.rtol = rtol
        self.atol = atol
        self.maxiter = maxiter
        self.estimate_cond = estimate_cond

    def setup(self, matrix, *args, **kwargs) -> None:
        if self.preconditioner is not None:
            self.preconditioner.setup(matrix, *args, **kwargs)

        self.matrix = matrix

    def solve(self, rhs, x0=None):
        atol = max(self.atol, self.rtol * np.linalg.norm(rhs).item())
        n = len(rhs)
        maxiter = self.maxiter or n * 10
        x = x0 if x0 is not None else np.zeros_like(rhs)
        if self.preconditioner is not None and hasattr(self.preconditioner, "T0"):
            x += self.preconditioner.T0(rhs)
        r = rhs - self.matrix @ x
        rho_prev, p = None, None

        residual_norms = []
        preconditioner_metadata = []
        if self.estimate_cond:
            alphas = []
            betas = []
        for i in range(maxiter):
            res_norm = np.linalg.norm(r).item()
            residual_norms.append(res_norm)
            if res_norm < atol:
                break

            (z, metadata) = (
                self.preconditioner.solve(r)
                if self.preconditioner is not None
                else (r, {})
            )

            preconditioner_metadata.append(metadata)
            rho_cur = np.dot(r, z)
            if i > 0:
                beta = rho_cur / rho_prev
                p *= beta
                p += z
            else:
                p = np.empty_like(r)
                p[:] = z[:]

            q = self.matrix @ p
            alpha = rho_cur / np.dot(p, q)
            x += alpha * p
            r -= alpha * q
            rho_prev = rho_cur

            if self.estimate_cond:
                if i > 0:
                    betas.append(beta)
                alphas.append(alpha)

        metadata = {
            "iterations": len(residual_norms) - 1,
            "residual norms": residual_norms,
            "preconditioner metadata": preconditioner_metadata,
        }

        if self.estimate_cond:
            lmat = np.zeros(
                (len(alphas), len(alphas)),
                dtype=rhs.dtype,
            )
            for i in range(len(alphas)):
                lmat[i, i] = 1 / alphas[i]
                if i > 0:
                    lmat[i, i] += betas[i - 1] / alphas[i - 1]
                    lmat[i - 1, i] = lmat[i, i - 1] = (
                        betas[i - 1] ** (1 / 2) / alphas[i - 1]
                    )
            metadata["condition number estimate"] = np.linalg.cond(lmat, p=2).item()

        if i + 1 == maxiter:
            raise Exception(metadata)

        return x, metadata

    def destroy(self) -> None:
        if self.preconditioner is not None:
            self.preconditioner.destroy()
            self.preconditioner = None
        self.matrix = None

In [52]:
class GMRES:
    name = "GMRES"

    def __init__(
        self,
        preconditioner=None,
        rtol: float = 1e-7,
        atol: float = 0,
        maxiter: int | None = 1000,
        estimate_cond: bool = False,  # ignored
    ):
        self.preconditioner = preconditioner
        self.rtol = rtol
        self.atol = atol
        self.maxiter = maxiter

    def setup(self, matrix, *args, **kwargs) -> None:
        if self.preconditioner is not None:
            self.preconditioner.setup(matrix, *args, **kwargs)

        self.matrix = matrix

    def solve(self, rhs, x0=None):
        M_x = lambda x: self.preconditioner.solve(x)[0] if self.preconditioner else x
        M = sps.linalg.LinearOperator(self.matrix.shape, matvec=M_x, dtype=rhs.dtype)

        residual_norms = []

        def callback(residual):
            residual_norms.append(np.linalg.norm(residual).item())

        x, info = sps.linalg.gmres(
            self.matrix,
            rhs,
            x0=x0,
            rtol=self.rtol,
            atol=self.atol,
            maxiter=self.maxiter,
            M=M,
            callback=callback,
            callback_type="legacy",
        )

        metadata = {
            "iterations": len(residual_norms),
            "residual norms": residual_norms,
        }

        if info > 0:
            raise Exception(metadata)

        return x, metadata

    def destroy(self) -> None:
        if self.preconditioner is not None:
            self.preconditioner.destroy()
            self.preconditioner = None
        self.matrix = None

In [53]:
class OverlappingSchwarz:
    def setup(
        self,
        matrix,
        fine_to_solvers: list[list[int]],
        solvers_to_fine: list[list[int]],
        R0T: sps.csr_matrix,
        dofs_per_element: int = 3,
    ):
        self.fine_to_solvers = fine_to_solvers
        self.solvers_to_fine = solvers_to_fine
        self.R0T = R0T
        self.dofs_per_element = dofs_per_element

        self.n_solvers = len(solvers_to_fine)

        self.solver_dofs = [[] for _ in range(self.n_solvers)]
        self.local_mult_matrices = [None] * self.n_solvers
        self.local_solvers = [None] * self.n_solvers
        for i in range(self.n_solvers):
            for el in solvers_to_fine[i]:
                for d in range(dofs_per_element):
                    self.solver_dofs[i].append(el * dofs_per_element + d)
            self.solver_dofs[i] = np.array(self.solver_dofs[i], dtype=np.int32)
            solver_matrix = matrix[self.solver_dofs[i]][:, self.solver_dofs[i]]
            self.local_mult_matrices[i] = matrix[self.solver_dofs[i]]
            self.local_solvers[i] = sps.linalg.splu(solver_matrix.tocsc())

        self.matrix = matrix

        coarse_matrix = R0T.T @ matrix @ R0T
        self.coarse_solver = sps.linalg.splu(coarse_matrix)

    def destroy(self):
        pass

    def R0(self, x: np.ndarray):
        return (
            x.reshape(-1, self.dofs_per_element)
            .sum(axis=1)[self.coarse_to_fine]
            .sum(axis=1)
        )

    def Ri(self, i: int, x: np.ndarray):
        return x[self.solver_dofs[i]]

    def RiT(self, i: int, y_local: np.ndarray, out_add: np.ndarray):
        out_add[self.solver_dofs[i]] += y_local

    def P0(self, x: np.ndarray):
        x_coarse = self.R0T.T @ x
        y_coarse = self.coarse_solver.solve(x_coarse)
        return self.R0T @ y_coarse

    def Pi(self, i: int, x: np.ndarray, out_add: np.ndarray):
        local_x = x[self.solver_dofs[i]]
        local_y = self.local_solvers[i].solve(local_x)
        out_add[self.solver_dofs[i]] += local_y

    def Mi(self, i: int, x: np.ndarray, y: np.ndarray):
        self.RiT(
            i,
            self.local_solvers[i].solve(
                self.Ri(i, x) - self.local_mult_matrices[i] @ y
            ),
            y,
        )

In [54]:
class AdditiveSchwarz(OverlappingSchwarz):
    symmetric = True
    name = "Additive"

    def solve(self, x: np.ndarray):
        y = self.P0(x)
        for i in range(self.n_solvers):
            self.Pi(i, x, y)
        return y, {}

In [55]:
class MultiplicativeSymmetricSchwarz(OverlappingSchwarz):
    symmetric = True
    name = "Multiplicative Symmetric"

    def solve(self, x: np.ndarray):
        y = self.P0(x)
        for i in range(self.n_solvers):
            self.Mi(i, x, y)
        for i in reversed(range(self.n_solvers)):
            self.Mi(i, x, y)
        return y, {}

In [56]:
class MultiplicativeSchwarz(OverlappingSchwarz):
    symmetric = False
    name = "Multiplicative"

    def solve(self, x: np.ndarray):
        y = self.P0(x)
        for i in range(self.n_solvers):
            self.Mi(i, x, y)
        return y, {}

In [57]:
class HybridSchwarz(OverlappingSchwarz):
    symmetric = True
    name = "Hybrid"

    def solve(self, x: np.ndarray):
        y = self.P0(x)
        z = x - self.matrix @ y
        u = np.zeros_like(x)
        for i in range(self.n_solvers):
            self.Pi(i, z, u)
        w = u - self.P0(self.matrix @ u)
        return y + w, {}

In [58]:
preconditioners = [
    AdditiveSchwarz,
    # MultiplicativeSymmetricSchwarz,
    # MultiplicativeSchwarz,
    # HybridSchwarz,
]

In [59]:
def fine_to_mesh(fine_mesh, n: int, N: int, overlap_n: int):
    v = fine_mesh.vertices[fine_mesh.elements]
    coords = (v.mean(axis=1) * n).astype(int)

    def coords_to_mesh(x, y):
        base_solver = (x // (n // N)) * N + (y // (n // N))
        in_solver_coords = (x % (n // N), y % (n // N))
        solvers = [base_solver]
        if in_solver_coords[1] < overlap_n and base_solver % N != 0:
            solvers.append(base_solver - 1)
        if in_solver_coords[1] >= (n // N - overlap_n) and base_solver % N != N - 1:
            solvers.append(base_solver + 1)
        if in_solver_coords[0] < overlap_n and base_solver >= N:
            solvers.append(base_solver - N)
        if in_solver_coords[0] >= (n // N - overlap_n) and base_solver < N * (N - 1):
            solvers.append(base_solver + N)
        # now we have to check the corners
        if (
            in_solver_coords[0] < overlap_n
            and in_solver_coords[1] < overlap_n
            and base_solver >= N
            and base_solver % N != 0
        ):
            solvers.append(base_solver - N - 1)
        if (
            in_solver_coords[0] < overlap_n
            and in_solver_coords[1] >= (n // N - overlap_n)
            and base_solver >= N
            and base_solver % N != N - 1
        ):
            solvers.append(base_solver - N + 1)
        if (
            in_solver_coords[0] >= (n // N - overlap_n)
            and in_solver_coords[1] < overlap_n
            and base_solver < N * (N - 1)
            and base_solver % N != 0
        ):
            solvers.append(base_solver + N - 1)
        if (
            in_solver_coords[0] >= (n // N - overlap_n)
            and in_solver_coords[1] >= (n // N - overlap_n)
            and base_solver < N * (N - 1)
            and base_solver % N != N - 1
        ):
            solvers.append(base_solver + N + 1)
        return solvers

    return [coords_to_mesh(x, y) for x, y in coords]

In [60]:
from dolfinx import fem, geometry, mesh
from petsc4py import PETSc


def assemble_prolongation_mixed_cells(V_coarse, V_fine):
    coarse_mesh = V_coarse.mesh
    fine_mesh = V_fine.mesh

    # 1. Get physical (x,y,z) coordinates of all fine DoFs
    points = V_fine.tabulate_dof_coordinates()
    num_fine_dofs = points.shape[0]

    # ------------------------------------------------------------------
    # 2. RESOLVE DG BOUNDARY AMBIGUITY (The Robust 5% Pull-back)
    # ------------------------------------------------------------------
    num_local_fine_cells = fine_mesh.topology.index_map(
        fine_mesh.topology.dim
    ).size_local
    fine_dof_to_cell = np.full(num_fine_dofs, -1, dtype=np.int32)
    fine_dofmap = V_fine.dofmap.list

    for c in range(num_local_fine_cells):
        fine_dof_to_cell[fine_dofmap[c]] = c

    # Safely get the exact centroids of all fine cells using a DG0 space
    V_dg0 = fem.functionspace(fine_mesh, ("DG", 0))
    dg0_coords = V_dg0.tabulate_dof_coordinates()
    dg0_dofmap = V_dg0.dofmap.list.flatten()

    fine_cell_centroids = np.zeros((num_local_fine_cells, 3))
    for c in range(num_local_fine_cells):
        fine_cell_centroids[c] = dg0_coords[dg0_dofmap[c]]

    # Pull the search points 5% inward towards their exact cell centroids.
    # This guarantees we escape FEniCS's bounding box collision tolerances!
    epsilon = 0.05
    cell_midpoints_for_dofs = fine_cell_centroids[fine_dof_to_cell]
    shifted_points = points + epsilon * (cell_midpoints_for_dofs - points)

    # ------------------------------------------------------------------
    # 3. Geometric Collision Search (Using the safely shifted points)
    # ------------------------------------------------------------------
    bb_tree = geometry.bb_tree(coarse_mesh, coarse_mesh.topology.dim)
    cell_candidates = geometry.compute_collisions_points(bb_tree, shifted_points)
    colliding_cells = geometry.compute_colliding_cells(
        coarse_mesh, cell_candidates, shifted_points
    )

    fine_to_coarse_cell = np.full(num_fine_dofs, -1, dtype=np.int32)
    for i in range(num_fine_dofs):
        links = colliding_cells.links(i)
        if len(links) > 0:
            fine_to_coarse_cell[i] = links[0]

    # Fallback for extreme outer boundaries
    missing_mask = fine_to_coarse_cell == -1
    if np.any(missing_mask):
        missing_points = shifted_points[missing_mask]
        midpoint_tree = geometry.create_midpoint_tree(
            coarse_mesh, coarse_mesh.topology.dim, bb_tree.links
        )
        closest_cells = geometry.compute_closest_entity(
            bb_tree, midpoint_tree, coarse_mesh, missing_points
        )
        fine_to_coarse_cell[missing_mask] = closest_cells

    # ------------------------------------------------------------------
    # 4. Probe coarse basis functions
    # ------------------------------------------------------------------
    B = V_coarse.dofmap.list.shape[1]
    basis_funcs_H = [fem.Function(V_coarse) for _ in range(B)]

    coarse_dofmap = V_coarse.dofmap.list
    num_local_coarse_cells = coarse_mesh.topology.index_map(
        coarse_mesh.topology.dim
    ).size_local

    for c in range(num_local_coarse_cells):
        for j in range(B):
            basis_funcs_H[j].x.array[coarse_dofmap[c, j]] = 1.0

    # ------------------------------------------------------------------
    # 5. Evaluate and Assemble
    # ------------------------------------------------------------------
    rows, cols, vals = [], [], []

    fine_global_dofs = V_fine.dofmap.index_map.local_to_global(
        np.arange(num_fine_dofs, dtype=np.int32)
    )
    coarse_global_map = V_coarse.dofmap.index_map.local_to_global(
        np.arange(V_coarse.dofmap.index_map.size_local, dtype=np.int32)
    )

    for j in range(B):
        # NOTE: We evaluate at the *original* points, using the corrected cell map!
        eval_vals = basis_funcs_H[j].eval(points, fine_to_coarse_cell).flatten()

        local_coarse_dofs = coarse_dofmap[fine_to_coarse_cell, j]
        global_cols = coarse_global_map[local_coarse_dofs]

        # Use 1e-10 to securely filter out DG polynomial evaluation float noise
        nonzero = np.abs(eval_vals) > 1e-10

        rows.append(fine_global_dofs[nonzero])
        cols.append(global_cols[nonzero])
        vals.append(eval_vals[nonzero])

    if len(rows) > 0:
        rows = np.concatenate(rows)
        cols = np.concatenate(cols)
        vals = np.concatenate(vals)

    # ------------------------------------------------------------------
    # 6. Build SciPy and PETSc Matrices
    # ------------------------------------------------------------------
    N_fine = V_fine.dofmap.index_map.size_global
    N_coarse = V_coarse.dofmap.index_map.size_global

    P_scipy = sps.csr_matrix((vals, (rows, cols)), shape=(N_fine, N_coarse))

    return P_scipy

In [61]:
import dolfinx
from mpi4py import MPI


def test(
    n,
    N,
    NN,
    overlap_n,
    coefficient="continuous",
    coefficient_param=None,
    coarse_elem="DG",
    coarse_deg=0,
):
    if N <= n:
        assert n % N == 0
        assert N % NN == 0
        assert overlap_n >= 0 and overlap_n <= n // N // 2
    else:
        assert N == n + 1
        assert n % NN == 0
        assert overlap_n == 0

    fine_mesh = Mesh2D.unit_square_uniform(n, n)
    coarse_mesh = Mesh2D(
        dolfinx.mesh.create_unit_square(
            comm=MPI.COMM_WORLD,
            nx=NN,
            ny=NN,
            cell_type=dolfinx.mesh.CellType.quadrilateral,
        )
    )

    fine_space = fem.functionspace(fine_mesh.dolfinx_mesh, ("DG", 1))
    coarse_space = fem.functionspace(
        coarse_mesh.dolfinx_mesh, (coarse_elem, coarse_deg)
    )
    R0T = assemble_prolongation_mixed_cells(coarse_space, fine_space)

    if N <= n:
        fine_to_solvers = fine_to_mesh(fine_mesh, n, N, overlap_n)
    else:
        fine_to_solvers = np.arange(fine_mesh.elements.shape[0], dtype=np.int32).reshape(-1, 1)

    solvers_to_fine = [[] for _ in range(N * N if N <= n else n * n * 2)]
    for i, solvers in enumerate(fine_to_solvers):
        for solver in solvers:
            solvers_to_fine[solver].append(i)

    if coefficient == "constant":
        problem = constant_coefficient_2d.problem
    elif coefficient == "continuous":
        problem = continuous_coefficient_2d.problem
    elif coefficient == "coarse checkerboard":
        problem = Problem(
            delta=7,
            rho=(
                lambda x: ((x[0] * NN).astype(int) + (x[1] * NN).astype(int))
                % 2
                * coefficient_param
                + 1
            ),
            f=constant_coefficient_2d.problem.f,
        )
    elif coefficient == "solvers checkerboard":
        problem = Problem(
            delta=7,
            rho=(
                lambda x: ((x[0] * N).astype(int) + (x[1] * N).astype(int))
                % 2
                * coefficient_param
                + 1
            ),
            f=constant_coefficient_2d.problem.f,
        )
    elif coefficient == "fine checkerboard":
        problem = Problem(
            delta=7,
            rho=(
                lambda x: ((x[0] * n).astype(int) + (x[1] * n).astype(int))
                % 2
                * coefficient_param
                + 1
            ),
            f=constant_coefficient_2d.problem.f,
        )
    elif coefficient == "madness":
        problem = Problem(
            delta=7,
            rho=(
                lambda x: (
                    # coeeficient_param on even and 1 on odd indices
                    np.vstack([np.full((x.shape[1] // 2), 1.0), np.full((x.shape[1] // 2), coefficient_param)]).T.flatten()
                )
            ),
            f=constant_coefficient_2d.problem.f,
        )
    elif coefficient == "madness2":
        problem = Problem(
            delta=7,
            rho=(
                lambda x: (
                    # coeeficient_param on even and 1 on odd indices
                    np.vstack([np.full((x.shape[1] // 2), 1.0), np.random.rand((x.shape[1] // 2)) * coefficient_param]).T.flatten()
                )
            ),
            f=constant_coefficient_2d.problem.f,
        )
    elif coefficient == "fine stripes":
        problem = Problem(
            delta=7,
            rho=(
                lambda x: ((x[0] * n).astype(int) % 3 == 0) * coefficient_param + 1
            ),
            f=constant_coefficient_2d.problem.f,
        )

    discrete_problem = discretize(
        problem=problem,
        mesh=fine_mesh,
    )

    results = []
    for preconditioner_cls in preconditioners:
        if preconditioner_cls.symmetric:
            # solvers = [GMRES, CG]
            solvers = [CG]
        else:
            solvers = [GMRES]

        for solver_cls in solvers:
            preconditioner = preconditioner_cls()
            solver = solver_cls(
                preconditioner=preconditioner,
                rtol=1e-6,
                maxiter=500,
                estimate_cond=True,
            )

            solver.setup(
                matrix=discrete_problem.exact_form_matrix,
                fine_to_solvers=fine_to_solvers,
                solvers_to_fine=solvers_to_fine,
                R0T=R0T,
                dofs_per_element=3,
            )

            test_metadata = {
                "n": n,
                "N": N,
                "NN": NN,
                "overlap_n": overlap_n,
                "preconditioner": preconditioner_cls.name,
                "solver": solver_cls.name,
                "coefficient": coefficient,
                "coefficient_param": coefficient_param,
                "coarse_elem": coarse_elem,
                "coarse_deg": coarse_deg,
            }

            try:
                rhs = np.random.rand(discrete_problem.load_vector.shape[0])
                rhs = (rhs - 0.5) * 2

                sol, metadata = solver.solve(rhs)
                residual_norm = np.linalg.norm(
                    discrete_problem.exact_form_matrix @ sol - rhs
                )
                results.append(
                    {
                        **test_metadata,
                        "residual norm": residual_norm,
                        "iterations": metadata["iterations"],
                        "condition number": metadata.get(
                            "condition number estimate", None
                        ),
                    }
                )
            except Exception as e:
                results.append(
                    {
                        **test_metadata,
                        "exception": str(e),
                    }
                )

            solver.destroy()

    return results

In [62]:
n = 2**5
NN = 2**3
Ns = [2**3, 2**4, n, n + 1]
coefficients = ["madness2"]
coefficient_params = [1, 1e2, 1e4, 1e6, 1e8]
coarse_elem = "DG"
coarse_deg = 0

test_cases = []
for N in Ns:
    overlaps = [0]
    if N <= n:
        overlap_n = 1
        while overlap_n <= n // N // 2:
            overlaps.append(overlap_n)
            overlap_n *= 2

    for overlap_n in overlaps:
        for coefficient in coefficients:
            for coefficient_param in coefficient_params:
                test_cases.append((n, N, NN, overlap_n, coefficient, coefficient_param, coarse_elem, coarse_deg))

len(test_cases)

35

In [63]:
results = []
for test_case in tqdm.tqdm(test_cases):
    results += test(*test_case)

100%|██████████| 35/35 [01:19<00:00,  2.29s/it]


In [70]:
df = pd.DataFrame(results)
# df.to_csv("overlapping_schwarz_results_coarse.csv", index=False)
# df[(df["coefficient"] == "solvers checkerboard")]
df[(df.coefficient == "madness2") & (df.preconditioner == "Additive")].pivot_table(
    values="condition number",
    index=["N", "overlap_n"],
    columns="coefficient_param",
)

coefficient_param  1.0          100.0         10000.0      1000000.0    \
N  overlap_n                                                             
8  0                 82.611736   347.709611  34831.024551          NaN   
   1                 50.386930    65.189648     74.340599    74.624600   
   2                 30.272436    36.273946     44.449537    44.667529   
16 0                 94.506946   506.199140  40732.087883          NaN   
   1                 61.955155    79.354191    101.759838   101.998715   
32 0                110.389754   506.931164           NaN          NaN   
33 0                137.763502   599.704133           NaN          NaN   

coefficient_param  100000000.0  
N  overlap_n                    
8  0                       NaN  
   1                 75.015074  
   2                 44.701902  
16 0                       NaN  
   1                101.997635  
32 0                       NaN  
33 0                       NaN

In [65]:
# jak amgx?

In [66]:
def get_order(row):
    h = 1 / row["n"]
    H = 1 / row["N"]
    HH = 1 / row["NN"]
    delta = row["overlap_n"] / row["n"]
    return HH * HH / H / max(h, delta)


df["estimate"] = df.apply(get_order, axis=1)
df["ratio"] = df["condition number"] / df["estimate"]
df[(df.N == df.NN)]

,n,N,NN,overlap_n,preconditioner,solver,coefficient,coefficient_param,coarse_elem,coarse_deg,residual norm,iterations,condition number,exception,estimate,ratio
0,32,8,8,0,Additive,CG,madness2,1.0,DG,0,0.000042,61.0,82.611736,NaN,4.0,20.652934
1,32,8,8,0,Additive,CG,madness2,100.0,DG,0,0.000042,124.0,347.709611,NaN,4.0,86.927403
2,32,8,8,0,Additive,CG,madness2,10000.0,DG,0,0.000041,462.0,34831.024551,NaN,4.0,8707.756138
3,32,8,8,0,Additive,CG,madness2,1000000.0,DG,0,NaN,NaN,NaN,"{'iterations': 499, 'residual norms': [44.9859...",4.0,NaN
4,32,8,8,0,Additive,CG,madness2,100000000.0,DG,0,NaN,NaN,NaN,"{'iterations': 499, 'residual norms': [45.0075...",4.0,NaN
5,32,8,8,1,Additive,CG,madness2,1.0,DG,0,0.000034,37.0,50.386930,NaN,4.0,12.596732
6,32,8,8,1,Additive,CG,madness2,100.0,DG,0,0.000034,42.0,65.189648,NaN,4.0,16.297412
7,32,8,8,1,Additive,CG,madness2,10000.0,DG,0,0.000039,47.0,74.340599,NaN,4.0,18.585150
8,32,8,8,1,Additive,CG,madness2,1000000.0,DG,0,0.000028,52.0,74.624600,NaN,4.0,18.656150
9,32,8,8,1,Additive,CG,madness2,100000000.0,DG,0,0.000032,55.0,75.015074,NaN,4.0,18.753769


In [67]:
def paper_test(
    n=7,
    NN=4,
    coarse_elem="DG",
    coarse_deg=0,
    coefficient_param=1e6,
):
    assert N % NN == 0

    fine_mesh = Mesh2D.unit_square_uniform(n, n)
    coarse_mesh = Mesh2D(
        dolfinx.mesh.create_unit_square(
            comm=MPI.COMM_WORLD,
            nx=NN,
            ny=NN,
            cell_type=dolfinx.mesh.CellType.quadrilateral,
        )
    )

    fine_space = fem.functionspace(fine_mesh.dolfinx_mesh, ("DG", 1))
    coarse_space = fem.functionspace(
        coarse_mesh.dolfinx_mesh, (coarse_elem, coarse_deg)
    )
    R0T = assemble_prolongation_mixed_cells(coarse_space, fine_space)

    fine_to_solvers = np.arange(fine_mesh.elements.shape[0])
    solvers_to_fine = np.arange(fine_mesh.elements.shape[0]).reshape(-1, 1)

    problem = Problem(
        delta=7,
        rho=(
            lambda x: (
                # coeeficient_param on even and 1 on odd indices
                np.vstack([np.full((x.shape[1] // 2), 1.0), np.full((x.shape[1] // 2), coefficient_param)]).T.flatten()
            )
        ),
        f=constant_coefficient_2d.problem.f,
    )

    discrete_problem = discretize(
        problem=problem,
        mesh=fine_mesh,
    )

    results = []
    for preconditioner_cls in preconditioners:
        if preconditioner_cls.symmetric:
            # solvers = [GMRES, CG]
            solvers = [CG]
        else:
            solvers = [GMRES]

        for solver_cls in solvers:
            preconditioner = preconditioner_cls()
            solver = solver_cls(
                preconditioner=preconditioner,
                rtol=1e-6,
                maxiter=2000,
                estimate_cond=True,
            )

            solver.setup(
                matrix=discrete_problem.exact_form_matrix,
                fine_to_solvers=fine_to_solvers,
                solvers_to_fine=solvers_to_fine,
                R0T=R0T,
                dofs_per_element=3,
            )

            test_metadata = {
                "n": n,
                "N": N,
                "NN": NN,
                # "overlap_n": overlap_n,
                "preconditioner": preconditioner_cls.name,
                "solver": solver_cls.name,
                "coefficient": coefficient,
                "coefficient_param": coefficient_param,
                "coarse_elem": coarse_elem,
                "coarse_deg": coarse_deg,
            }

            try:
                rhs = np.random.rand(discrete_problem.load_vector.shape[0])
                rhs = (rhs - 0.5) * 2

                sol, metadata = solver.solve(rhs)
                residual_norm = np.linalg.norm(
                    discrete_problem.exact_form_matrix @ sol - rhs
                )
                results.append(
                    {
                        **test_metadata,
                        "residual norm": residual_norm,
                        "iterations": metadata["iterations"],
                        "condition number": metadata.get(
                            "condition number estimate", None
                        ),
                    }
                )
            except Exception as e:
                results.append(
                    {
                        **test_metadata,
                        "exception": str(e),
                    }
                )

            solver.destroy()

    return results

In [68]:
paper_test(n=2**4, NN=2**3, coefficient_param=10000)

AssertionError: 